In [1]:
## Import thư viện

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

In [9]:
df = pd.read_csv('/Users/nguyentien/Documents/dhv-handout2-salary/dhv-handout2-salary/data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.csv')
df = df.copy()

# Đặt tên cột ngắn — tên gốc dài hàng chục từ, rất khó dùng
df.columns = [
    'timestamp',        # Timestamp
    'age',              # How old are you?
    'industry',         # What industry do you work in?
    'job_title',        # Job title
    'job_context',      # If your job title needs additional context...
    'annual_salary',    # What is your annual salary?
    'add_compensation', # How much additional monetary compensation...
    'currency',         # Please indicate the currency
    'currency_other',   # If "Other," please indicate the currency here
    'income_context',   # If your income needs additional context...
    'country',          # What country do you work in?
    'us_state',         # If you're in the U.S., what state do you work in?
    'city',             # What city do you work in?
    'years_exp_total',  # How many years of professional work experience overall?
    'years_exp_field',  # How many years of professional work experience in field?
    'education',        # What is your highest level of education completed?
    'gender',           # What is your gender?
    'race'              # What is your race?
]

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df.shape)
print(df.dtypes)
df.head()

Shape: (28216, 18)
Columns: ['timestamp', 'age', 'industry', 'job_title', 'job_context', 'annual_salary', 'add_compensation', 'currency', 'currency_other', 'income_context', 'country', 'us_state', 'city', 'years_exp_total', 'years_exp_field', 'education', 'gender', 'race']
(28216, 18)
timestamp            object
age                  object
industry             object
job_title            object
job_context          object
annual_salary        object
add_compensation    float64
currency             object
currency_other       object
income_context       object
country              object
us_state             object
city                 object
years_exp_total      object
years_exp_field      object
education            object
gender               object
race                 object
dtype: object


,timestamp,age,industry,job_title,job_context,annual_salary,add_compensation,currency,currency_other,income_context,country,us_state,city,years_exp_total,years_exp_field,education,gender,race
0,4/27/2021 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,"55,000",0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,4/27/2021 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,"34,000",NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White
3,4/27/2021 11:02:41,25-34,Nonprofits,Program Manager,NaN,"62,000",3000.0,USD,NaN,NaN,USA,Wisconsin,Milwaukee,8 - 10 years,5-7 years,College degree,Woman,White
4,4/27/2021 11:02:42,25-34,"Accounting, Banking & Finance",Accounting Manager,NaN,"60,000",7000.0,USD,NaN,NaN,US,South Carolina,Greenville,8 - 10 years,5-7 years,College degree,Woman,White


## TUẦN 1 — Missing Values, Duplicates & Data Types

In [8]:
def missing_report(df):
    """Báo cáo số lượng và % missing của từng cột có null."""
    miss = df.isnull().sum()
    pct  = (miss / len(df) * 100).round(2)
    return (pd.DataFrame({'count': miss, 'pct': pct})
              .query('count > 0')
              .sort_values('pct', ascending=False))

print("MISSING VALUES REPORT")
print(missing_report(df))

MISSING VALUES REPORT
                  count    pct
currency_other    27993  99.21
income_context    25163  89.18
job_context       20924  74.16
add_compensation   7372  26.13
us_state           5071  17.97
education           240   0.85
race                196   0.69
gender              185   0.66
industry             85   0.30
city                 85   0.30
job_title             4   0.01
country               2   0.01


### 1.1 Phân tích Missing Values

| Cột | % Null | Loại Null | Lý do | Chiến lược |
|-----|--------|-----------|-------|------------|
| `currency_other` | 99.2% | Null hợp lệ | Chỉ điền khi chọn "Other" currency | Giữ nguyên — không xử lý |
| `income_context` | 89.2% | Null hợp lệ | Free-text tùy chọn | Giữ nguyên |
| `job_context` | 74.2% | Null hợp lệ | Free-text tùy chọn | Giữ nguyên |
| `add_compensation` | 26.1% | MCAR | Không phải ai cũng có bonus | fillna(0) — không có bonus = 0 |
| `us_state` | 18.0% | Null hợp lệ | Chỉ dành cho người ở Mỹ | Giữ nguyên |
| `education` | 0.85% | MCAR | Bỏ qua ngẫu nhiên | fillna('Unknown') |
| `gender` | 0.66% | MCAR | Không muốn khai | fillna('Prefer not to answer') |
| `city` | 0.30% | MCAR | Bỏ qua ngẫu nhiên | fillna('Unknown') |
| `industry` | 0.30% | MCAR | Bỏ qua ngẫu nhiên | fillna('Unknown') |
| `country` | 0.01% | MCAR | 2 hàng bỏ trống | Drop 2 hàng này |

**Phân tích MCAR vs MNAR:**
- **MCAR** (Missing Completely At Random): thiếu hoàn toàn ngẫu nhiên — an toàn fillna
- **MNAR** (Missing Not At Random): `add_compensation` null có nghĩa là $0 bonus thực sự — đây là giá trị có ý nghĩa, không phải "không biết" → fillna(0) là đúng
- **Null hợp lệ**: `currency_other`, `job_context`, `income_context`, `us_state` — null vì câu hỏi không áp dụng cho họ, không phải lỗi

In [10]:
## Xử lý missing values
null_before = df.isnull().sum().sum()

# LÝ DO: currency_other, income_context, job_context, us_state
# là null HỢP LỆ — câu hỏi không áp dụng cho mọi người → KHÔNG xử lý

# LÝ DO: add_compensation null = không có bonus = 0 (MNAR có ý nghĩa rõ ràng)
df['add_compensation'] = df['add_compensation'].fillna(0)

# LÝ DO: education/gender/city/industry null MCAR → điền Unknown
df['education'] = df['education'].fillna('Unknown')
df['gender']    = df['gender'].fillna('Prefer not to answer')
df['city']      = df['city'].fillna('Unknown')
df['industry']  = df['industry'].fillna('Unknown')

# LÝ DO: country null chỉ có 2 hàng → drop vì không biết xử lý thế nào
df = df.dropna(subset=['country']).reset_index(drop=True)

# So sánh
null_after = df.isnull().sum().sum()
print(f"Null trước : {null_before:,}")
print(f"Null sau   : {null_after:,}")
print(f"Đã xử lý  : {null_before - null_after:,} giá trị")
print("\n=== CÒN LẠI (null hợp lệ) ===")
print(missing_report(df))

Null trước : 87,320
Null sau   : 79,342
Đã xử lý  : 7,978 giá trị

=== CÒN LẠI (null hợp lệ) ===
                count    pct
currency_other  27991  99.21
income_context  25161  89.18
job_context     20922  74.15
us_state         5070  17.97
race              195   0.69
job_title           3   0.01


### 1.2 Kiểm tra Duplicates

**Kết quả: 0 duplicate toàn hàng.**

Lý do có thể không có duplicate:
- Google Forms tự ngăn submit 2 lần trong cùng session
- Mỗi submission có Timestamp riêng → khó trùng hoàn toàn

Tuy nhiên vẫn cần kiểm tra **duplicate theo key columns** —
cùng người có thể submit 2 lần tại 2 thời điểm khác nhau.

In [11]:
# Duplicate toàn hàng
n_dup = df.duplicated().sum()
print(f"Duplicate toàn hàng: {n_dup}")

# Duplicate theo "soft key" — cùng job_title + industry + annual_salary + country
# (vì không có email/user_id để identify)
key_dup = df.duplicated(subset=['job_title', 'industry',
                                 'annual_salary', 'country']).sum()
print(f"Duplicate theo key (job+industry+salary+country): {key_dup}")

# Xem thử các hàng "gần duplicate"
dupes = df[df.duplicated(subset=['job_title', 'industry',
                                  'annual_salary', 'country'], keep=False)]
print(f"\nSố hàng có thể trùng: {len(dupes)}")
print(dupes[['timestamp','job_title','industry','annual_salary','country']].head(6))

Duplicate toàn hàng: 0
Duplicate theo key (job+industry+salary+country): 353

Số hàng có thể trùng: 662
              timestamp           job_title                       industry  \
3    4/27/2021 11:02:41     Program Manager                     Nonprofits   
132  4/27/2021 11:05:46     Product Manager              Computing or Tech   
211  4/27/2021 11:06:50  Associate Attorney                            Law   
250  4/27/2021 11:07:18             teacher  Education (Primary/Secondary)   
378  4/27/2021 11:09:02    Business Analyst              Computing or Tech   
379  4/27/2021 11:09:03             teacher  Education (Primary/Secondary)   

    annual_salary        country  
3          62,000            USA  
132       130,000  United States  
211       125,000  United States  
250        94,000  United States  
378        85,000  United States  
379        94,000  United States  


### 1.3 Phân tích Data Types

**Vấn đề dtype thực tế trong dataset này:**

| Cột | Dtype hiện tại | Dtype đúng | Vấn đề |
|-----|---------------|------------|--------|
| `timestamp` | object (string) | datetime64 | Không thể tính khoảng thời gian, extract year/month |
| `annual_salary` | object (string) | float64 | Có dấu phẩy "55,000" → không tính được mean/median |
| `industry` | object | category | Tốn RAM — chỉ có ~30 giá trị unique trong 28k hàng |
| `age` | object (range) | string/ordinal | Là range "25-34", không phải số — phải xử lý đặc biệt |
| `years_exp_total` | object (range) | string/ordinal | Tương tự age — "5-7 years" |

**Lưu ý đặc biệt về cột `age` và `years_exp`:**  
Đây là **range string** như "25-34", "5-7 years" — KHÔNG convert sang số được
bằng astype(float). Phải dùng mapping dict để convert sang ordinal hoặc
lấy điểm giữa (midpoint) của mỗi khoảng.

In [12]:
print("=== DTYPE TRƯỚC ===")
print(df.dtypes)
print("\nVí dụ salary:", df['annual_salary'].head(5).tolist())
print("Ví dụ age:", df['age'].unique().tolist())

# --- Convert annual_salary: "55,000" → 55000.0 ---
# LÝ DO: có dấu phẩy hàng nghìn → không tính được toán học
df['annual_salary'] = (df['annual_salary']
    .astype(str)
    .str.replace(',', '', regex=False)   # "55,000" → "55000"
    .str.strip())
df['annual_salary'] = pd.to_numeric(df['annual_salary'], errors='coerce')
# errors='coerce': nếu có giá trị lạ như "N/A" → thành NaN thay vì crash

# --- Convert timestamp ---
# LÝ DO: cần extract năm/tháng để làm feature survey_year
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# --- Convert industry sang category (tiết kiệm RAM) ---
# LÝ DO: ~30 giá trị unique / 28k hàng → category tiết kiệm 5-10x RAM
mem_before = df['industry'].memory_usage(deep=True)
df['industry'] = df['industry'].astype('category')
mem_after  = df['industry'].memory_usage(deep=True)
print(f"\nRAM tiết kiệm (industry): {(mem_before-mem_after)/1024:.0f} KB")

# --- Convert age range → ordinal number (điểm giữa của range) ---
# LÝ DO: cần số để phân tích thống kê; dùng midpoint là ước tính hợp lý
age_map = {
    'under 18': 17,
    '18-24'   : 21,
    '25-34'   : 29,
    '35-44'   : 39,
    '45-54'   : 49,
    '55-64'   : 59,
    '65 or over': 70
}
df['age_midpoint'] = df['age'].map(age_map)

# --- Convert years_exp range → ordinal (midpoint) ---
# LÝ DO: tương tự age — cần số để tính salary_per_exp
exp_map = {
    '1 year or less': 0.5,
    '2 - 4 years'   : 3,
    '5-7 years'      : 6,
    '8 - 10 years'   : 9,
    '11 - 20 years'  : 15,
    '21 - 30 years'  : 25,
    '31 - 40 years'  : 35,
    '41 years or more': 45
}
df['years_exp_field_num']  = df['years_exp_field'].map(exp_map)
df['years_exp_total_num']  = df['years_exp_total'].map(exp_map)

print("\n=== DTYPE SAU ===")
print(df.dtypes)

=== DTYPE TRƯỚC ===
timestamp            object
age                  object
industry             object
job_title            object
job_context          object
annual_salary        object
add_compensation    float64
currency             object
currency_other       object
income_context       object
country              object
us_state             object
city                 object
years_exp_total      object
years_exp_field      object
education            object
gender               object
race                 object
dtype: object

Ví dụ salary: ['55,000', '54,600', '34,000', '62,000', '60,000']
Ví dụ age: ['25-34', '45-54', '35-44', '18-24', '65 or over', '55-64', 'under 18']

RAM tiết kiệm (industry): 1933 KB

=== DTYPE SAU ===
timestamp              datetime64[ns]
age                            object
industry                     category
job_title                      object
job_context                    object
annual_salary                   int64
add_compensation              f